# 2 - MLP Call Option Pricer

A 4-layer fully-connected network (400 units/layer, LeakyReLU, BatchNorm) trained to predict call closing prices directly from contract features.

Volatility is re-estimated as a 20-day rolling standard deviation of returns (`sigma_20`) rather than using the dataset's supplied `sigma`.

Trained for 200 epochs at lr=1e-5, then fine-tuned at successively lower learning rates. **Note the first stage is the best one** - see the README, later stages degrade.

In [2]:
# @title Default title text
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, LeakyReLU, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

In [3]:
# Hyperparams
n_units = 400
layers = 4
n_batch = 4096
n_epochs = 200

In [5]:
data= pd.read_excel('../data/ASIANPAINT_Dataset.xlsx')
estimate_σ = lambda arr: (np.diff(arr) / arr[:-1]).std()
data['sigma_20'] = data.close.rolling(20).apply(estimate_σ)
data.dropna(subset=['sigma_20'], inplace=True)

data2=data.copy(deep=True)
data.head()

,Date,Expiry,t,strike_price,underlying_value,sigma,r,close,sigma_20
19,2020-01-01,2020-01-30,29,1660,1793.2,0.008151,0.0494,104.90,23.727248
20,2020-01-01,2020-01-30,29,1680,1793.2,0.008151,0.0494,136.00,1.405593
21,2020-01-01,2020-01-30,29,1560,1793.2,0.008151,0.0494,252.00,1.351294
22,2020-01-01,2020-01-30,29,1580,1793.2,0.008151,0.0494,279.15,1.012983
23,2020-01-01,2020-01-30,29,1600,1793.2,0.008151,0.0494,225.05,1.011580


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 35572 entries, 19 to 35590
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              35572 non-null  datetime64[ns]
 1   Expiry            35572 non-null  datetime64[ns]
 2   t                 35572 non-null  int64         
 3   strike_price      35572 non-null  int64         
 4   underlying_value  35572 non-null  float64       
 5   sigma             35572 non-null  float64       
 6   r                 35572 non-null  float64       
 7   close             35572 non-null  float64       
 8   sigma_20          35572 non-null  float64       
dtypes: datetime64[ns](2), float64(5), int64(2)
memory usage: 2.7 MB


In [114]:

data = data[data.underlying_value > data.strike_price]
data2 = data2[data2.underlying_value < data2.strike_price]

data.head(30)

,Date,Expiry,t,strike_price,underlying_value,sigma,r,close,sigma_20
19,2020-01-01,2020-01-30,29,1660,1793.2,0.008151,0.0494,104.90,23.727248
20,2020-01-01,2020-01-30,29,1680,1793.2,0.008151,0.0494,136.00,1.405593
21,2020-01-01,2020-01-30,29,1560,1793.2,0.008151,0.0494,252.00,1.351294
22,2020-01-01,2020-01-30,29,1580,1793.2,0.008151,0.0494,279.15,1.012983
23,2020-01-01,2020-01-30,29,1600,1793.2,0.008151,0.0494,225.05,1.011580
24,2020-01-01,2020-01-30,29,1520,1793.2,0.008151,0.0494,230.00,1.017040
25,2020-01-01,2020-01-30,29,1540,1793.2,0.008151,0.0494,311.50,0.985234
26,2020-01-01,2020-01-30,29,1460,1793.2,0.008151,0.0494,380.50,0.973408
27,2020-01-01,2020-01-30,29,1480,1793.2,0.008151,0.0494,331.00,0.968192
28,2020-01-01,2020-01-30,29,1500,1793.2,0.008151,0.0494,320.85,0.669144


In [115]:
call_X_train, call_X_test, call_y_train, call_y_test = train_test_split(data.drop(['close','Date','Expiry','sigma'], axis=1),
                                                                        (data.close),
                                                                        test_size=0.2, random_state=42)
put_X_train, put_X_test, put_y_train, put_y_test = train_test_split(data2.drop(['close','Date','Expiry','sigma'], axis=1),
                                                                    (data2.close),
                                                                      test_size=0.2, random_state=42)

In [116]:
model = Sequential()
model.add(BatchNormalization(input_shape=(call_X_train.shape[1],)))
model.add(Dense(n_units, input_dim=call_X_train.shape[1]))
model.add(LeakyReLU())

for _ in range(layers - 1):
    model.add(Dense(n_units))
    model.add(BatchNormalization())
    model.add(LeakyReLU())

model.add(Dense(1, activation='relu'))

model.compile(loss='mse', optimizer=Adam(lr=1e-5))

In [117]:
model.summary()

Model: "sequential_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 batch_normalization_30 (Ba  (None, 5)                 20        
 tchNormalization)                                               
                                                                 
 dense_40 (Dense)            (None, 400)               2400      
                                                                 
 leaky_re_lu_32 (LeakyReLU)  (None, 400)               0         
                                                                 
 dense_41 (Dense)            (None, 400)               160400    
                                                                 
 batch_normalization_31 (Ba  (None, 400)               1600      
 tchNormalization)                                               
                                                                 
 leaky_re_lu_33 (LeakyReLU)  (None, 400)              

In [118]:
history = model.fit(call_X_train, call_y_train,
                    batch_size=n_batch, epochs=n_epochs,
                    validation_split = 0.01,
                    callbacks=[TensorBoard()],
                    verbose=1)

Epoch 1/200
5/5 [==============================] - 5s 473ms/step - loss: 114009.5859 - val_loss: 107730.9453
Epoch 2/200
5/5 [==============================] - 2s 362ms/step - loss: 110112.2422 - val_loss: 107300.3438
Epoch 3/200
5/5 [==============================] - 2s 381ms/step - loss: 108720.9531 - val_loss: 107238.7500
Epoch 4/200
5/5 [==============================] - 1s 234ms/step - loss: 107884.6094 - val_loss: 107689.1641
Epoch 5/200
5/5 [==============================] - 1s 234ms/step - loss: 107146.2031 - val_loss: 107592.2969
Epoch 6/200
5/5 [==============================] - 1s 234ms/step - loss: 106399.0078 - val_loss: 107552.1953
Epoch 7/200
5/5 [==============================] - 1s 239ms/step - loss: 105629.3750 - val_loss: 107646.1250
Epoch 8/200
5/5 [==============================] - 1s 233ms/step - loss: 104853.9688 - val_loss: 107611.0859
Epoch 9/200
5/5 [==============================] - 1s 232ms/step - loss: 104062.7031 - val_loss: 107591.7891
Epoch 10/200
5/5 [=

In [119]:
model.save('mlp1-call10.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [120]:
call_y_pred = model.predict(call_X_test)

135/135 [==============================] - 1s 4ms/step


In [121]:
#import sklearn.metrics
from sklearn.metrics import mean_squared_error,mean_absolute_error
accuracy = mean_squared_error(call_y_pred,call_y_test)
print(accuracy)
accuracy1=mean_absolute_error(call_y_pred,call_y_test)
print(accuracy1)
print(np.sqrt(accuracy))

2978.2472551293454
38.87140752105315
54.573319993650244


In [122]:
model.compile(loss='mse', optimizer=Adam(lr=1e-6))

In [123]:
history = model.fit(call_X_train, call_y_train,
                    batch_size=n_batch, epochs=20,
                    validation_split = 0.01,
                    callbacks=[TensorBoard()],
                    verbose=1)

Epoch 1/20
5/5 [==============================] - 4s 443ms/step - loss: 7074.0850 - val_loss: 64648.5312
Epoch 2/20
5/5 [==============================] - 2s 381ms/step - loss: 5876.9668 - val_loss: 62534.3516
Epoch 3/20
5/5 [==============================] - 2s 385ms/step - loss: 4880.6792 - val_loss: 37830.2734
Epoch 4/20
5/5 [==============================] - 1s 261ms/step - loss: 4133.3950 - val_loss: 26861.6426
Epoch 5/20
5/5 [==============================] - 1s 235ms/step - loss: 3899.6978 - val_loss: 9623.2822
Epoch 6/20
5/5 [==============================] - 1s 237ms/step - loss: 3716.3926 - val_loss: 18300.4609
Epoch 7/20
5/5 [==============================] - 1s 238ms/step - loss: 3701.7368 - val_loss: 21548.5859
Epoch 8/20
5/5 [==============================] - 1s 238ms/step - loss: 3602.7708 - val_loss: 13349.3291
Epoch 9/20
5/5 [==============================] - 1s 239ms/step - loss: 3278.6973 - val_loss: 6985.8750
Epoch 10/20
5/5 [==============================] - 1s 236

In [124]:
model.save('mlp1-20.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [125]:
call_y_pred = model.predict(call_X_test)


135/135 [==============================] - 1s 4ms/step


In [126]:
from sklearn.metrics import mean_squared_error,mean_absolute_error
accuracy = mean_squared_error(call_y_pred,call_y_test)
print(accuracy)
accuracy1=mean_absolute_error(call_y_pred,call_y_test)
print(accuracy1)
print(np.sqrt(accuracy))

5843.742325130048
57.57952327613211
76.44437405806949


In [127]:
model.compile(loss='mse', optimizer=Adam(lr=1e-7))

In [128]:
history = model.fit(call_X_train, call_y_train,
                    batch_size=n_batch, epochs=10,
                    validation_split = 0.01,
                    callbacks=[TensorBoard()],
                    verbose=1)

Epoch 1/10
5/5 [==============================] - 4s 301ms/step - loss: 13672.5469 - val_loss: 65864.0859
Epoch 2/10
5/5 [==============================] - 1s 250ms/step - loss: 8942.6514 - val_loss: 85582.6250
Epoch 3/10
5/5 [==============================] - 2s 414ms/step - loss: 7075.4097 - val_loss: 59312.6641
Epoch 4/10
5/5 [==============================] - 3s 458ms/step - loss: 6665.4751 - val_loss: 46143.0078
Epoch 5/10
5/5 [==============================] - 2s 390ms/step - loss: 6297.8486 - val_loss: 42435.8828
Epoch 6/10
5/5 [==============================] - 1s 241ms/step - loss: 5607.5640 - val_loss: 40785.0000
Epoch 7/10
5/5 [==============================] - 1s 242ms/step - loss: 5287.7290 - val_loss: 33382.3320
Epoch 8/10
5/5 [==============================] - 1s 243ms/step - loss: 4975.8989 - val_loss: 21675.0332
Epoch 9/10
5/5 [==============================] - 1s 241ms/step - loss: 4727.1201 - val_loss: 12786.6738
Epoch 10/10
5/5 [==============================] - 1s 

In [129]:
model.save('mlp1-115.h5')
call_y_pred = model.predict(call_X_test)

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


135/135 [==============================] - 1s 4ms/step


In [130]:
from sklearn.metrics import mean_squared_error,mean_absolute_error
accuracy = mean_squared_error(call_y_pred,call_y_test)
print(accuracy)
accuracy1=mean_absolute_error(call_y_pred,call_y_test)
print(accuracy1)
print(np.sqrt(accuracy))

9518.883431880435
74.31079492936831
97.56476531966054


In [131]:
model.compile(loss='mse', optimizer=Adam(lr=1e-8))
history = model.fit(call_X_train, call_y_train,
                    batch_size=n_batch, epochs=5,
                    validation_split = 0.01,
                    callbacks=[TensorBoard()],
                    verbose=1)

Epoch 1/5
5/5 [==============================] - 4s 454ms/step - loss: 5799.0923 - val_loss: 12574.2656
Epoch 2/5
5/5 [==============================] - 2s 390ms/step - loss: 4762.8506 - val_loss: 13109.7627
Epoch 3/5
5/5 [==============================] - 2s 401ms/step - loss: 4415.3433 - val_loss: 5543.5688
Epoch 4/5
5/5 [==============================] - 1s 258ms/step - loss: 3653.2351 - val_loss: 5112.6968
Epoch 5/5
5/5 [==============================] - 1s 240ms/step - loss: 3262.7354 - val_loss: 6412.1426


In [134]:
model.save('mlp1-120.h5')
call_y_pred = model.predict(call_X_test)

  1/135 [..............................] - ETA: 3s

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


135/135 [==============================] - 1s 5ms/step


In [133]:
from sklearn.metrics import mean_squared_error,mean_absolute_error
accuracy = mean_squared_error(call_y_pred,call_y_test)
print(accuracy)
accuracy1=mean_absolute_error(call_y_pred,call_y_test)
print(accuracy1)
print(np.sqrt(accuracy))

6189.307232760062
62.14751914918285
78.67215029958227
